# Robust AI-Generated Image Detection — Colab Runner

This notebook is a **thin orchestrator**. It contains no model logic.
Everything real lives in `src/` in the repo and is version-controlled:

| module | responsibility |
|---|---|
| `src/transforms.py` | the 15 degradation transforms (JPEG, blur, resize, noise, jitter, crop) |
| `src/features.py` | frozen DINOv2 CLS + radial FFT profile, fused to 800-d |
| `src/model.py` | the classifier head, save/load |
| `src/train.py` | training with a real train/val split (test held out) |
| `src/evaluate.py` | robustness sweep → `outputs/robustness_table.csv` |
| `src/inference.py` | folder → `predictions.json` |

Colab's only jobs here are **providing the GPU** and **running the commands below**.
If you want to change behaviour, edit the code in `src/` and push — not this notebook.

> **Set the runtime to GPU first:** Runtime → Change runtime type → T4 GPU.
> Switching the runtime wipes the session, so do it before running anything.

## 0. Confirm the GPU

Feature extraction is the expensive step (one DINOv2 forward pass per image, and
the robustness sweep repeats that 15 times). On CPU this goes from minutes to hours,
so check before committing to a run.

In [ ]:
!nvidia-smi || echo 'No GPU visible — set Runtime -> Change runtime type -> T4 GPU, then re-run from the top.'

## 1. Get the code

Clones on first run, pulls on later runs, so re-running the notebook picks up
whatever was last pushed.

In [ ]:
import os

REPO_URL = "https://github.com/Adxtxp/TikTok-Hack.git"
REPO_DIR = "/content/TikTok-Hack"

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    !cd {REPO_DIR} && git pull --ff-only
else:
    !git clone {REPO_URL} {REPO_DIR}

# All later cells run relative to the repo root, which is what the
# `python -m src.<module>` invocations below expect.
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())
!git log --oneline -1

## 2. Install dependencies

Colab already ships torch/torchvision built against its CUDA driver, which is why
they are left unpinned in `requirements.txt` — pip should leave the preinstalled
build alone rather than swapping it for a CPU wheel.

`albumentations>=2.0` matters: the transforms use the `quality_range` and
`std_range` argument names, which replaced the 1.x spellings.

In [ ]:
!pip install -q -r requirements.txt

import albumentations, torch
print("albumentations", albumentations.__version__, "(needs >= 2.0)")
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

## 3. Download CIFAKE and build the manifest

The only data-wrangling step that stays in the notebook, because it depends on
Colab-specific paths and a Kaggle download.

The manifest is the contract every `src/` script reads — one row per image:

| column | meaning |
|---|---|
| `image_path` | absolute path to the file |
| `label` | `0` = real, `1` = AI-generated |
| `split` | `train` or `test`, taken from the dataset's own directory layout |

Label and split are inferred from the CIFAKE directory names
(`train/REAL`, `train/FAKE`, `test/REAL`, `test/FAKE`). The dataset's own
train/test division is preserved and never re-drawn — `src/train.py` carves its
validation set out of the `train` rows only, so the `test` rows stay untouched
until `src/evaluate.py`.

In [ ]:
import csv
from pathlib import Path

import kagglehub
from tqdm.auto import tqdm

path = kagglehub.dataset_download("birdy654/cifake-real-and-ai-generated-synthetic-images")
print("Path to dataset files:", path)

DATA_DIR = Path(path)  # contains train/REAL, train/FAKE, test/REAL, test/FAKE
OUTPUT_CSV = Path("/content/manifest.csv")
IMAGE_EXTS = {".jpg", ".jpeg", ".png"}


def guess_label(p: Path):
    parts = [x.lower() for x in p.parts]
    if "real" in parts:
        return 0
    if "fake" in parts:
        return 1
    return None


def guess_split(p: Path):
    parts = [x.lower() for x in p.parts]
    if "train" in parts:
        return "train"
    if "test" in parts:
        return "test"
    return "unknown"


# rglob over ~120k files is slow and silent. No total is known up front, so
# the bar shows a running count and rate rather than a percentage.
all_images = [
    p
    for p in tqdm(DATA_DIR.rglob("*"), desc="Scanning dataset files", unit="file")
    if p.suffix.lower() in IMAGE_EXTS
]
print(f"Found {len(all_images)} images under {DATA_DIR}\n")

rows, unknown = [], []
for img in tqdm(all_images, desc="Labeling images", unit="img"):
    label = guess_label(img)
    if label is None:
        unknown.append(img)
    else:
        rows.append((str(img), label, guess_split(img)))

real_count = sum(1 for _, l, _ in rows if l == 0)
fake_count = sum(1 for _, l, _ in rows if l == 1)
print(f"Auto-labeled: {real_count} real, {fake_count} fake, {len(unknown)} unknown")

with open(OUTPUT_CSV, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["image_path", "label", "split"])  # label: 0 = real, 1 = fake
    writer.writerows(rows)

print(f"Wrote manifest for {len(rows)} labeled images to {OUTPUT_CSV}")

In [ ]:
# Sanity-check the manifest before spending GPU time on it.
import pandas as pd

MANIFEST = "/content/manifest.csv"
df = pd.read_csv(MANIFEST)
print(df.shape)
print(df.groupby(["split", "label"]).size())
df.head()

### Optional: work on a subset first

CIFAKE is 120k images. A full clean training pass means 100k DINOv2 forward
passes, and `--augment` re-extracts on top of that. Run the subset first to
confirm the pipeline end-to-end, then rerun on the full manifest.

Sampling is per (split, label), so the train/test division and class balance
both survive.

In [ ]:
# Subset size. These defaults are chosen so the notebook runs top-to-bottom on
# a stock Colab T4 without editing anything.
USE_SUBSET = True     # False = use the full dataset
PER_GROUP = 15000     # images per (split, label) group

# Written by the manifest-building cell above. Named explicitly so this cell
# never depends on a value left over from an earlier run.
FULL_MANIFEST = "/content/manifest.csv"

if USE_SUBSET:
    # min(PER_GROUP, len(g)) caps each group by its own size, so the
    # 10k-per-class test groups are taken whole while the 50k-per-class train
    # groups are sampled down to PER_GROUP.
    sub = (
        df.groupby(["split", "label"], group_keys=False)
          .apply(lambda g: g.sample(n=min(PER_GROUP, len(g)), random_state=42))
    )
    MANIFEST = "/content/manifest_subset.csv"
    sub.to_csv(MANIFEST, index=False)
    print(f"subset -> {MANIFEST}")
    print(sub.groupby(["split", "label"]).size())
    n_train = int((sub["split"] == "train").sum())
    n_test = int((sub["split"] == "test").sum())
    print()
    print(f"total {len(sub)} rows: {n_train} train, {n_test} test")
    print(f"each extraction pass runs ~{n_train} DINOv2 forwards - the "
          f"dominant cost of everything below.")
else:
    # Assigned explicitly rather than inherited from the sanity-check cell.
    # Without this line, flipping USE_SUBSET to False and re-running JUST
    # this cell would leave MANIFEST pointing at the subset CSV written by a
    # previous run, while printing that the full manifest is in use.
    MANIFEST = FULL_MANIFEST
    print(f"using full manifest -> {MANIFEST}")
    print(df.groupby(["split", "label"]).size())
    print()
    print(f"total {len(df)} rows - expect a long extraction. Back up to "
          f"Drive (cell below) before the session can drop.")

## 4. Train the head

Runs `src/train.py`. Only `split == "train"` rows are read; a stratified
`val_split` (0.15 from `configs/default.yaml`) is carved out of those for the
per-epoch metrics. **The test split is never loaded here** — the validation
numbers printed are not a test score.

Add `--augment` to apply a random degradation per training image before feature
extraction (validation stays clean). That is the run to use for the robustness
story; it is slower, because features can't be cached across transforms.

In [ ]:
!python -u -m src.train \
  --manifest {MANIFEST} \
  --out outputs/head_fused.pt \
  --epochs 30 \
  --cache-embeddings outputs/embeddings_fused.npz

In [ ]:
# Robustness-trained variant. Compare its robustness table against the clean
# head's to show what augmentation actually bought.
!python -u -m src.train \
  --manifest {MANIFEST} \
  --out outputs/head_augmented.pt \
  --epochs 30 \
  --augment

### Back up `outputs/` to Google Drive

Feature extraction and training are the expensive steps here, and a Colab
disconnect wipes the VM — taking `outputs/` (and `.gitignore`d artifacts like
the trained heads, scalers and cached embeddings) with it.

Run this now, and re-run it any time afterwards — e.g. again after the
evaluation cells — to pick up newly written files. Each run writes a
timestamped folder plus a `latest/` mirror.


In [ ]:
# Copy everything produced so far into Drive, so a crash doesn't cost a
# re-run of the expensive extraction/training steps.
import os
import shutil
from datetime import datetime

BACKUP_ROOT = "/content/drive/MyDrive/tiktok-hack-outputs"

try:
    from google.colab import drive
    drive.mount("/content/drive")  # no-op if already mounted
    mounted = True
except Exception as exc:
    print(f"Drive unavailable ({type(exc).__name__}: {exc}) - skipping backup. "
          f"Nothing else in the notebook depends on this cell.")
    mounted = False

if mounted:
    # .gitkeep is a placeholder committed to the repo, not a real artifact.
    artifacts = []
    for root, _dirs, files in os.walk("outputs"):
        artifacts += [os.path.join(root, f) for f in files if f != ".gitkeep"]

    if not artifacts:
        print("nothing in outputs/ to back up yet - run the training cells first")
    else:
        stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
        dest = os.path.join(BACKUP_ROOT, stamp)
        latest = os.path.join(BACKUP_ROOT, "latest")

        # dirs_exist_ok merges into any existing folder. Deliberately no
        # rmtree first: this writes into your real Drive, so it only ever adds
        # or overwrites files by name and never deletes anything already there.
        shutil.copytree("outputs", dest, dirs_exist_ok=True)
        shutil.copytree("outputs", latest, dirs_exist_ok=True)

        total = sum(os.path.getsize(p) for p in artifacts)
        print(f"backed up {len(artifacts)} file(s), {total / 1e6:.1f} MB -> {dest}")
        for p in sorted(artifacts):
            print(f"  {os.path.relpath(p, 'outputs')}  "
                  f"({os.path.getsize(p) / 1e6:.1f} MB)")
        print(f"also mirrored to {latest}")


## 5. Evaluate robustness (the graded table)

Runs `src/evaluate.py` on a class-balanced sample of the **held-out test
split** — 500 per class by default. Sweeps all 15 transforms, writes
`outputs/robustness_table.csv`, and prints a confusion-matrix diagnosis for
`clean`, `blur_2.0`, `resize_0.25` and `noise_0.10`.

This is the first and only point at which the test split is touched.

In [ ]:
!python -u -m src.evaluate \
  --manifest {MANIFEST} \
  --head outputs/head_fused.pt \
  --out outputs/robustness_table.csv

In [ ]:
# Same sweep for the augmented head, so the two tables can be compared.
!python -u -m src.evaluate \
  --manifest {MANIFEST} \
  --head outputs/head_augmented.pt \
  --out outputs/robustness_table_augmented.csv \
  --skip-diagnose

In [ ]:
# Side-by-side: did augmented training reduce the drop under degradation?
import pandas as pd

base = pd.read_csv("outputs/robustness_table.csv", index_col=0)
try:
    aug = pd.read_csv("outputs/robustness_table_augmented.csv", index_col=0)
    cmp = pd.DataFrame({
        "acc_clean_head": base["accuracy"],
        "acc_augmented_head": aug["accuracy"],
    })
    cmp["improvement"] = cmp["acc_augmented_head"] - cmp["acc_clean_head"]
    display(cmp.round(4))
except FileNotFoundError:
    print("augmented table not found — run the --augment cells above first")
    display(base.round(4))

## 6. Inference on a folder of unlabelled images

Runs `src/inference.py`. Point `--image_dir` at any folder — upload your own,
or use the cell below to stage a few test images. Writes a JSON list of
`{"image_path": ..., "pred": ...}`, where `pred` is P(AI-generated).

In [ ]:
# Stage a small demo folder from the test split, keeping a ground-truth table
# so the predictions can be scored below.
import shutil
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm

DEMO_DIR = Path("/content/my_test_images")
shutil.rmtree(DEMO_DIR, ignore_errors=True)
DEMO_DIR.mkdir(parents=True)

demo = (
    pd.read_csv(MANIFEST)
      .query("split == 'test'")
      .groupby("label", group_keys=False)
      .apply(lambda g: g.sample(n=5, random_state=0))
)

gt_rows = []
for i, r in enumerate(tqdm(list(demo.itertuples()), desc="Copying demo images", unit="img")):
    dest = DEMO_DIR / f"img_{i:02d}{Path(r.image_path).suffix}"
    shutil.copy(r.image_path, dest)
    # inference.py writes the path it was given, so record the same string.
    gt_rows.append({"image_path": str(dest), "label": r.label})

ground_truth = pd.DataFrame(gt_rows)
print(f"staged {len(ground_truth)} images in {DEMO_DIR}")
ground_truth

In [ ]:
!python -u -m src.inference \
  --image_dir /content/my_test_images \
  --weights outputs/head_fused.pt \
  --output outputs/predictions.json

In [ ]:
# Score the predictions against the staged ground truth.
import json

import pandas as pd

with open("outputs/predictions.json") as f:
    preds_df = pd.DataFrame(json.load(f))

merged = preds_df.merge(ground_truth, on="image_path")
if len(merged) != len(preds_df):
    print(f"WARNING: only {len(merged)}/{len(preds_df)} rows merged — "
          "image_path strings differ between the two sides")

merged["predicted_label"] = (merged["pred"] > 0.5).astype(int)
merged["correct"] = merged["predicted_label"] == merged["label"]

print(merged[["image_path", "label", "pred", "predicted_label", "correct"]].to_string(index=False))
print(f"\nAccuracy on this sample: {merged['correct'].mean():.2%}")

## 7. Save the artifacts

`outputs/` is gitignored, and a Colab session is wiped when it disconnects.
Download anything you want to keep.

In [ ]:
from google.colab import files

for artifact in [
    "outputs/robustness_table.csv",
    "outputs/predictions.json",
    "outputs/head_fused.pt",
]:
    try:
        files.download(artifact)
    except Exception as exc:
        print(f"skip {artifact}: {exc}")